In [1]:
import requests
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine
import psycopg2
from kafka import KafkaProducer
from kafka import KafkaConsumer
import os
import json

In [2]:
load_dotenv()

True

In [3]:
API_KEY  = os.getenv("API_KEY")
API_HOST = os.getenv("API_HOST")
LANG     = os.getenv("LANG")
Postgres_URI = os.getenv("POSTGRES_URI")

In [5]:
# url = f'https://open-weather13.p.rapidapi.com/city?city=city&lang=EN' 
Cities = ['Nairobi','Accra','Cape Town','Riga','Brussels','Moscow','Seoul','London','Sucre']

for city in Cities:
    url = f"https://{API_HOST}/city?city={city}&lang={LANG}"

    headers = {'x-rapidapi-host': API_HOST,
                   'x-rapidapi-key' : API_KEY
                   }
    
    response = requests.get(url, headers=headers)

    data = response.json()
    print(data)


{'coord': {'lon': 36.8167, 'lat': -1.2833}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'broken clouds', 'icon': '04d'}], 'base': 'stations', 'main': {'temp': 72.72, 'feels_like': 72.5, 'temp_min': 72.72, 'temp_max': 72.72, 'pressure': 1014, 'humidity': 60, 'sea_level': 1014, 'grnd_level': 837}, 'visibility': 10000, 'wind': {'speed': 11.5, 'deg': 90}, 'clouds': {'all': 75}, 'dt': 1779031261, 'sys': {'type': 1, 'id': 2543, 'country': 'KE', 'sunrise': 1778988450, 'sunset': 1779031857}, 'timezone': 10800, 'id': 184745, 'name': 'Nairobi', 'cod': 200}
{'coord': {'lon': -0.1969, 'lat': 5.556}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02d'}], 'base': 'stations', 'main': {'temp': 88.21, 'feels_like': 100.81, 'temp_min': 88.21, 'temp_max': 88.21, 'pressure': 1010, 'humidity': 74, 'sea_level': 1010, 'grnd_level': 1007}, 'visibility': 10000, 'wind': {'speed': 19.57, 'deg': 190}, 'clouds': {'all': 20}, 'dt': 1779031234, 'sys': {'type': 1, 'id': 1

In [6]:
Cities = ['Nairobi','Accra','Cape Town','Riga','Brussels','Moscow','Seoul','London','Sucre']

for city in Cities:
    url = f"https://{API_HOST}/city?city={city}&lang={LANG}"

    headers = {'x-rapidapi-host': API_HOST,
                   'x-rapidapi-key' : API_KEY
                   }
    
    response = requests.get(url, headers=headers)

    data = response.json()

    transformed_data = {
            "city" : data.get("name"),
            "temperature" : data.get("main",{}).get("temp"),
            "humidity" : data.get("main",{}).get("humidity"),
            "pressure" : data.get("main",{}).get("pressure"),
            "weather" : data.get("weather",[{}])[0].get("main"),
            "description" : data.get("weather",[{}])[0].get("description"),
            "wind_speed" : data.get("wind",{}).get("speed")
        }
    print(transformed_data)

{'city': 'Nairobi', 'temperature': 72.72, 'humidity': 60, 'pressure': 1014, 'weather': 'Clouds', 'description': 'broken clouds', 'wind_speed': 11.5}
{'city': 'Accra', 'temperature': 88.21, 'humidity': 74, 'pressure': 1010, 'weather': 'Clouds', 'description': 'few clouds', 'wind_speed': 19.57}
{'city': 'Cape Town', 'temperature': 65.01, 'humidity': 81, 'pressure': 1018, 'weather': 'Clear', 'description': 'clear sky', 'wind_speed': 15.99}
{'city': 'Rīga', 'temperature': 57.04, 'humidity': 53, 'pressure': 1016, 'weather': 'Clear', 'description': 'clear sky', 'wind_speed': 9.22}
{'city': 'Brussels', 'temperature': 53.89, 'humidity': 87, 'pressure': 1012, 'weather': 'Rain', 'description': 'shower rain', 'wind_speed': 12.66}
{'city': 'Moscow', 'temperature': 77.43, 'humidity': 48, 'pressure': 1012, 'weather': 'Rain', 'description': 'light rain', 'wind_speed': 9.89}
{'city': 'Seoul', 'temperature': 60.37, 'humidity': 77, 'pressure': 1018, 'weather': 'Clear', 'description': 'clear sky', 'wind_

In [7]:
Cities = ['Nairobi','Accra','Cape Town','Riga','Brussels','Moscow','Seoul','London','Sucre']

for city in Cities:
    url = f"https://{API_HOST}/city?city={city}&lang={LANG}"

    headers = {'x-rapidapi-host': API_HOST,
                   'x-rapidapi-key' : API_KEY
                   }
    
    response = requests.get(url, headers=headers)

    data = response.json()

    transformed_data = {
            "city" : data.get("name"),
            "temperature" : data.get("main",{}).get("temp"),
            "humidity" : data.get("main",{}).get("humidity"),
            "pressure" : data.get("main",{}).get("pressure"),
            "weather" : data.get("weather",[{}])[0].get("main"),
            "description" : data.get("weather",[{}])[0].get("description"),
            "wind_speed" : data.get("wind",{}).get("speed")
        }
    
    df = pd.DataFrame([transformed_data])
    print(df)

      city  temperature  humidity  pressure weather    description  wind_speed
0  Nairobi        72.72        60      1014  Clouds  broken clouds        11.5
    city  temperature  humidity  pressure weather description  wind_speed
0  Accra        88.21        74      1010  Clouds  few clouds       19.57
        city  temperature  humidity  pressure weather description  wind_speed
0  Cape Town        65.01        81      1018   Clear   clear sky       15.99
   city  temperature  humidity  pressure weather description  wind_speed
0  Rīga        57.04        53      1016   Clear   clear sky        9.22
       city  temperature  humidity  pressure weather  description  wind_speed
0  Brussels        53.89        87      1012    Rain  shower rain       12.66
     city  temperature  humidity  pressure weather description  wind_speed
0  Moscow        77.43        48      1012    Rain  light rain        9.89
    city  temperature  humidity  pressure weather description  wind_speed
0  Seoul    

In [8]:
Cities = ['Nairobi','Accra','Cape Town','Riga','Brussels','Moscow','Seoul','London','Sucre']

for city in Cities:
    url = f"https://{API_HOST}/city?city={city}&lang={LANG}"

    headers = {'x-rapidapi-host': API_HOST,
                   'x-rapidapi-key' : API_KEY
                   }
    
    response = requests.get(url, headers=headers)

    data = response.json()

    transformed_data = {
            "city" : data.get("name"),
            "temperature" : data.get("main",{}).get("temp"),
            "humidity" : data.get("main",{}).get("humidity"),
            "pressure" : data.get("main",{}).get("pressure"),
            "weather" : data.get("weather",[{}])[0].get("main"),
            "description" : data.get("weather",[{}])[0].get("description"),
            "wind_speed" : data.get("wind",{}).get("speed")
        }
    
    df = pd.json_normalize(transformed_data)
    print(df.head())

      city  temperature  humidity  pressure weather    description  wind_speed
0  Nairobi        72.72        60      1014  Clouds  broken clouds        11.5
    city  temperature  humidity  pressure weather description  wind_speed
0  Accra        88.21        74      1010  Clouds  few clouds       19.57
        city  temperature  humidity  pressure weather description  wind_speed
0  Cape Town        65.01        81      1018   Clear   clear sky       15.99
   city  temperature  humidity  pressure weather description  wind_speed
0  Rīga        57.04        53      1016   Clear   clear sky        9.22
       city  temperature  humidity  pressure weather  description  wind_speed
0  Brussels        53.89        87      1012    Rain  shower rain       12.66
     city  temperature  humidity  pressure weather description  wind_speed
0  Moscow        77.43        48      1012    Rain  light rain        9.89
    city  temperature  humidity  pressure weather description  wind_speed
0  Seoul    

In [9]:
Cities = ['Nairobi','Accra','Cape Town','Riga','Brussels','Moscow','Seoul','London','Sucre']

engine = create_engine(Postgres_URI)

for city in Cities:
    url = f"https://{API_HOST}/city?city={city}&lang={LANG}"

    headers = {'x-rapidapi-host': API_HOST,
                   'x-rapidapi-key' : API_KEY
                   }
    
    response = requests.get(url, headers=headers)

    data = response.json()

    transformed_data = {
            "city" : data.get("name"),
            "temperature" : data.get("main",{}).get("temp"),
            "humidity" : data.get("main",{}).get("humidity"),
            "pressure" : data.get("main",{}).get("pressure"),
            "weather" : data.get("weather",[{}])[0].get("main"),
            "description" : data.get("weather",[{}])[0].get("description"),
            "wind_speed" : data.get("wind",{}).get("speed")
        }
    
    df = pd.json_normalize(transformed_data)

    df.to_sql("weather_kafka", con=engine, if_exists="append",index=False)

    print(f"Loaded weather data for {transformed_data['city']}")



Loaded weather data for Nairobi
Loaded weather data for Accra
Loaded weather data for Cape Town
Loaded weather data for Rīga
Loaded weather data for Brussels
Loaded weather data for Moscow
Loaded weather data for Seoul
Loaded weather data for London
Loaded weather data for Sucre
